In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tensorflow.keras import Input
import seaborn as sns
from xgboost import XGBRegressor
from prophet import Prophet
from sklearn.linear_model import LinearRegression

In [ ]:
df = pd.read_parquet('/content/all_features.parquet')
df

FileNotFoundError: [Errno 2] No such file or directory: '/content/all_features.parquet'

In [ ]:
cpi = pd.read_csv('/content/CPIAUCSL.csv')
unrate = pd.read_csv('/content/UNRATE.csv')
mortgage = pd.read_csv('/content/MORTGAGE30US.csv')

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
cpi["observation_date"] = pd.to_datetime(cpi["observation_date"])
unrate["observation_date"] = pd.to_datetime(unrate["observation_date"])
mortgage["observation_date"] = pd.to_datetime(mortgage["observation_date"])

In [ ]:
cpi.rename(columns={"observation_date":"Date", "CPIAUCSL": "cpi"}, inplace=True)
unrate.rename(columns={"observation_date":"Date", "UNRATE": "unrate"}, inplace=True)
mortgage.rename(columns={"observation_date":"Date", "MORTGAGE30US": "mortgage_rate"}, inplace=True)

In [ ]:
mortgage

In [ ]:
# Function to normalize any date to the 1st day of its month
def normalize_to_month_start(df, date_col):
    """Converts a datetime column to the first day of the respective month."""
    # .dt.to_period('M') extracts the Month/Year period,
    # and .dt.to_timestamp() converts it back to the start of the month's timestamp
    return df[date_col].dt.to_period('M').dt.to_timestamp()

In [ ]:
df['Date'] = normalize_to_month_start(df, 'Date')
df

In [ ]:
df = pd.merge(
    df,
    cpi,
    on='Date',
    how='left'
)
df

In [ ]:
df = pd.merge(
    df,
    unrate,
    on='Date',
    how='left'
)
df

In [ ]:
mortgage.set_index('Date', inplace=True)
mortgage = mortgage.resample('M').mean()
mortgage.reset_index(inplace=True)
mortgage

In [ ]:
mortgage['Date'] = normalize_to_month_start(mortgage, 'Date')
mortgage

In [ ]:
df = pd.merge(
    df,
    mortgage,
    on='Date',
    how='left'
)
df

In [ ]:
df["Month"] = pd.to_datetime(df["Date"]).dt.month
df["Month_Sin"] = np.sin(2 * np.pi * df["Month"]/12)
df["Month_Cos"] = np.cos(2 * np.pi * df["Month"]/12)

In [ ]:
df

In [ ]:
df["HomeValue_to_Income"] = df["HomeValue"] / df["IncomeNeeded"]
df["MortgageBurden"] = df["mortgage_rate"] * df["HomeValue_to_Income"]

In [ ]:
df

In [ ]:
df["Inventory_to_Sales"] = df["Inventory"] / (df["SalesCount"] + 1)

In [ ]:
df

In [ ]:
df["HomeValue_Change"] = df.groupby("RegionName")["HomeValue"].pct_change().fillna(0)
df["IncomeNeeded_Change"] = df.groupby("RegionName")["IncomeNeeded"].pct_change().fillna(0)
df["CPI_Change"] = df["cpi"].pct_change().fillna(0)
df["MortgageRate_Change"] = df["mortgage_rate"].pct_change().fillna(0)
df["Unemployment_Change"] = df["unrate"].pct_change().fillna(0)

In [ ]:
df

In [ ]:
for col in ["cpi","mortgage_rate","unrate"]:
    df[f"{col}_lag1"] = df[col].shift(1).fillna(0)
    df[f"{col}_roll3"] = df[col].rolling(3).mean().fillna(0)

In [ ]:
df.to_parquet("all_features_with_macro.parquet")

In [ ]:
def evaluate_metro(df, model, features, params=None, test_months=24 ):
  train = df.iloc[:-test_months]
  test  = df.iloc[-test_months:]
  target   = "IncomeNeeded"
  X_train, y_train = train[features], train[target]
  X_test,  y_test  = test[features],  test[target]
  evaluate_model = model(**params) if params else model()
  evaluate_model.fit(X_train, y_train)
  pred = evaluate_model.predict(X_test)
  mae  = mean_absolute_error(y_test, pred)
  rmse = np.sqrt(mean_squared_error(y_test, pred))
  return mae, rmse

In [ ]:
results_xgb = []
params = {
    "n_estimators": 200,
    "learning_rate": 0.1,
    "max_depth": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
}
features = ['HomeValue', 'Inventory', 'DaysToPending', 'RentValue',
       'RenterIncomeNeeded', 'MarketHeatIndex', 'SalesCount',
       'NewConstruction', 'HomeValue_lag1', 'Inventory_lag1', 'RentValue_lag1',
       'MarketHeatIndex_lag1', 'HomeValue_roll3', 'Inventory_roll3',
       'RentValue_roll3', 'MarketHeatIndex_roll3', 'cpi', 'unrate',
       'mortgage_rate', 'Month', 'Month_Sin', 'Month_Cos',
       'HomeValue_to_Income', 'MortgageBurden', 'Inventory_to_Sales',
       'HomeValue_Change', 'CPI_Change',
       'MortgageRate_Change', 'Unemployment_Change', 'cpi_lag1', 'cpi_roll3',
       'mortgage_rate_lag1', 'mortgage_rate_roll3', 'unrate_lag1',
       'unrate_roll3']

for region, df_region in df.groupby("RegionID"):
  df_region = df_region.sort_values("Date").dropna()
  if len(df_region) < 24:
    continue
  MODEL_CLASS =  XGBRegressor
  mae, rmse = evaluate_metro(df_region, MODEL_CLASS, features, params)
  results_xgb.append({
    "RegionID": region,
    "RegionName": df_region["RegionName"].iloc[0],
    "StateName": df_region["StateName"].iloc[0],
    "MAE_xgb": mae,
    "RMSE_xgb": rmse,
  })

results_xgb_df = pd.DataFrame(results_xgb)
results_xgb_df

In [ ]:
results_xgb_df[["MAE_xgb", "RMSE_xgb"]].mean()

In [ ]:
results_xgb_df.to_csv("xgb_validation.csv", index=False)

In [ ]:
#LSTM

def prepare_lstm_data(df, feature_cols, target_col="IncomeNeeded", lookback=12):
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df[feature_cols + [target_col]])
    X, y = [], []
    for i in range(lookback, len(scaled)):
        X.append(scaled[i-lookback:i, :-1])
        y.append(scaled[i, -1])
    X, y = np.array(X), np.array(y)
    return X, y, scaler

In [ ]:
def build_lstm_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

In [ ]:
# def forecast_lstm_for_metro(df_metro, feature_cols, target_col="IncomeNeeded", lookback=12, forecast_horizon=12, epochs=20, batch_size=8):
#     # Prepare data
#     X, y, scaler = prepare_lstm_data(df_metro, feature_cols, target_col, lookback)

#     # Build and train model
#     model = build_lstm_model((X.shape[1], X.shape[2]))
#     model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=0)

#     # Start forecasting using the last window
#     last_sequence = X[-1:]
#     future_predictions = []

#     for _ in range(forecast_horizon):
#         next_scaled = model.predict(last_sequence, verbose=0)[0, 0]
#         future_predictions.append(next_scaled)

#         # Slide window and add new prediction
#         next_features = last_sequence[:, 1:, :].copy()
#         new_timestep = np.zeros((1, 1, last_sequence.shape[2]))
#         new_timestep[0, 0, -1] = next_scaled
#         last_sequence = np.concatenate([next_features, new_timestep], axis=1)

#     # Inverse scaling
#     y_pred_full = np.hstack([
#         np.zeros((len(future_predictions), len(feature_cols))),
#         np.array(future_predictions).reshape(-1, 1)
#     ])
#     y_pred_rescaled = scaler.inverse_transform(y_pred_full)[:, -1]

#     # Generate future dates
#     forecast_dates = pd.date_range(df_metro["Date"].max(), periods=forecast_horizon+1, freq="M")[1:]

#     # Return forecast DataFrame
#     return pd.DataFrame({
#         "Date": forecast_dates,
#         "RegionName": df_metro["RegionName"].iloc[0],
#         "ForecastedIncomeNeeded": y_pred_rescaled
#     })

In [ ]:
# forecast_list = []
# features = ['HomeValue', 'Inventory', 'DaysToPending', 'RentValue',
#        'RenterIncomeNeeded', 'MarketHeatIndex', 'SalesCount',
#        'NewConstruction', 'HomeValue_lag1', 'Inventory_lag1', 'RentValue_lag1',
#        'MarketHeatIndex_lag1', 'HomeValue_roll3', 'Inventory_roll3',
#        'RentValue_roll3', 'MarketHeatIndex_roll3', 'cpi', 'unrate',
#        'mortgage_rate', 'Month', 'Month_Sin', 'Month_Cos',
#        'HomeValue_to_Income', 'MortgageBurden', 'Inventory_to_Sales',
#        'HomeValue_Change', 'CPI_Change',
#        'MortgageRate_Change', 'Unemployment_Change', 'cpi_lag1', 'cpi_roll3',
#        'mortgage_rate_lag1', 'mortgage_rate_roll3', 'unrate_lag1',
#        'unrate_roll3']

# for metro in df["RegionName"].unique():
#     df_metro = df[df["RegionName"] == metro].sort_values("Date")
#     if len(df_metro) < 50:  # skip small metros
#         continue

#     forecast_df = forecast_lstm_for_metro(df_metro, features, forecast_horizon=24)
#     forecast_list.append(forecast_df)

# lstm_forecasts = pd.concat(forecast_list, ignore_index=True)
# lstm_forecasts = lstm_forecasts.sort_values("Date")
# lstm_forecasts.to_csv("lstm_forecasts.csv", index=False)

In [ ]:
lstm_forecasts

In [ ]:
def evaluate_lstm_all_metros(df, feature_cols, target_col="IncomeNeeded", test_months=12,
                             lookback=12, epochs=20, batch_size=8):
    results = []

    for metro in df["RegionName"].unique():
        df_metro = df[df["RegionName"] == metro].sort_values("Date").copy()

        if len(df_metro) < (lookback + test_months + 1):
            continue

        # Prepare data
        X, y, scaler = prepare_lstm_data(df_metro, feature_cols, target_col, lookback)

        # Train/test split
        X_train, X_test = X[:-test_months], X[-test_months:]
        y_train, y_test = y[:-test_months], y[-test_months:]

        # Build model
        model = build_lstm_model((X.shape[1], X.shape[2]))

        # Fit model
        model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)

        # Predict
        y_pred = model.predict(X_test, verbose=0)

        # Reconstruct arrays for inverse scaling
        y_test_full = np.hstack([np.zeros((len(y_test), len(feature_cols))), y_test.reshape(-1,1)])
        y_pred_full = np.hstack([np.zeros((len(y_pred), len(feature_cols))), y_pred])

        # Inverse scale
        y_test_rescaled = scaler.inverse_transform(y_test_full)[:, -1]
        y_pred_rescaled = scaler.inverse_transform(y_pred_full)[:, -1]

        # Metrics
        mae = mean_absolute_error(y_test_rescaled, y_pred_rescaled)
        rmse = np.sqrt(mean_squared_error(y_test_rescaled, y_pred_rescaled))

        results.append({
            "RegionName": metro,
            "LSTM_MAE": mae,
            "LSTM_RMSE": rmse
        })

    return pd.DataFrame(results)

In [ ]:
lstm_results = evaluate_lstm_all_metros(
    df=df,
    feature_cols=features,
    target_col="IncomeNeeded",
    test_months=12,
    lookback=12,
    epochs=20,
    batch_size=8
)
lstm_results.to_csv("lstm_validation.csv", index=False)

In [ ]:
lstm_results

In [ ]:
lstm_results[['LSTM_MAE',	'LSTM_RMSE']].mean()

In [ ]:
def evaluate_metro(df, model, features, params=None, test_months=12 ):
  train = df.iloc[:-test_months]
  test  = df.iloc[-test_months:]
  target   = "IncomeNeeded"
  X_train, y_train = train[features], train[target]
  X_test,  y_test  = test[features],  test[target]
  evaluate_model = model(**params) if params else model()
  evaluate_model.fit(X_train, y_train)
  pred = evaluate_model.predict(X_test)
  mae  = mean_absolute_error(y_test, pred)
  rmse = np.sqrt(mean_squared_error(y_test, pred))
  return mae, rmse

In [ ]:
linear_features = [
    'HomeValue', 'Inventory', 'DaysToPending', 'RentValue',
    'MarketHeatIndex', 'SalesCount', 'NewConstruction',
    'cpi', 'unrate', 'mortgage_rate',
    'Month_Sin', 'Month_Cos',
    'HomeValue_to_Income', 'MortgageBurden', 'Inventory_to_Sales',
    'HomeValue_Change', 'CPI_Change', 'MortgageRate_Change', 'Unemployment_Change'
]

results = []

for region, df_region in df.groupby("RegionID"):
  if len(df_region) > 15:
    MODEL_CLASS = LinearRegression
    mae, rmse = evaluate_metro(df_region, MODEL_CLASS, linear_features)
    results.append({
        "RegionID": region,
        "RegionName": df_region["RegionName"].iloc[0],
        "StateName": df_region["StateName"].iloc[0],
        "MAE": mae,
        "RMSE": rmse,
    })

results_df = pd.DataFrame(results)

In [ ]:
results_df[['MAE','RMSE']].mean()

In [ ]:
results_df.to_csv("linear_validation.csv", index=False)

In [ ]:
rent_error_table = pd.concat([results_df["MAE", "RMSE"], lstm_results['LSTM_MAE',	'LSTM_RMSE'], results_xgb_df["MAE_xgb", "RMSE_xgb"]], axis=1)
rent_error_table

NameError: name 'results_df' is not defined